### Check order of StructDef

1. Establish the resource
2. create or fetch a key order list for the type
3. create a key order list for the Struct.
4. compare

#### Get a Resource and Create a path list
  - This is a first pass TODOs
    - iterate over types if complex and expand
    - repeating logic

In [96]:
from pathlib import Path
from json import loads,dumps
from pprint import pprint

my_path = Path(r'/Users/ehaas/.fhir/packages/hl7.fhir.r4.core#4.0.1/package/')
my_resource = 'AllergyIntolerance'
line = 0
path_list = []

In [97]:
def limit_recursion(max_depth):
    def decorator(func):
        func._depth = 0
        def wrapper(*args, **kwargs):
            func._depth += 1
            if func._depth > max_depth:
                func._depth -= 1
                # print("recursion depth limit reached")
                return None
            try:
                return func(*args, **kwargs)
            finally:
                func._depth -= 1
        return wrapper
    return decorator

In [99]:
def get_my_file(type_name):
  # print(f"type_name = {type_name}")
  my_file = my_path / f"StructureDefinition-{type_name}.json"
  # print(f"my_file = {my_file}")
  try:
    fhir_obj = loads(my_file.read_text())
  except FileNotFoundError as e:
    fhir_obj = None
  return fhir_obj


def get_types(e):
  e_type =  e.get('type')
  try:
    my_types = [type['code'] for type in e_type]
  except TypeError:
      my_types = None
  return my_types


@limit_recursion(3)
def get_path(i, fhir_obj,old_path=None):
  global line, my_resource, path_list
  line +=1
  # print(f"old_path={old_path}")
  type_names = get_types(fhir_obj)
  # break
  try:
    new_path = fhir_obj['path'].split('.',1)[1]
  except IndexError as e:
    return None
  # print(new_path)
  if old_path:
    current_path = f"{old_path}.{new_path}"
  else:
    current_path = new_path
  # print(current_path)
  # print(f"{line}: {my_resource}.{current_path}, {fhir_obj['max']=='*'}, {type_names}")
  path_list.append((f"{my_resource}.{current_path}", fhir_obj['max']=='*', type_names))
  old_path = current_path
  try:
    for type_name in type_names:
      fhir_obj = get_my_file(type_name)
      # print(dumps(my_obj, indent=2))
      try:
        for j,e in enumerate(fhir_obj['snapshot']['element']):
          get_path(i+j,e,old_path)
      except KeyError as e:
        pass
        # print(f"no Snapshot for {type_name}")



  except TypeError:
      pass
  return

fhir_obj = get_my_file(my_resource)
for i,e in enumerate(fhir_obj['snapshot']['element']):
  get_path(i,e)
print(line, len(path_list))
path_list

4382 3712


[('AllergyIntolerance.id', False, ['http://hl7.org/fhirpath/System.String']),
 ('AllergyIntolerance.meta', False, ['Meta']),
 ('AllergyIntolerance.meta.id',
  False,
  ['http://hl7.org/fhirpath/System.String']),
 ('AllergyIntolerance.meta.extension', True, ['Extension']),
 ('AllergyIntolerance.meta.extension.id',
  False,
  ['http://hl7.org/fhirpath/System.String']),
 ('AllergyIntolerance.meta.extension.extension', True, ['Extension']),
 ('AllergyIntolerance.meta.extension.url',
  False,
  ['http://hl7.org/fhirpath/System.String']),
 ('AllergyIntolerance.meta.extension.value[x]',
  False,
  ['base64Binary',
   'boolean',
   'canonical',
   'code',
   'date',
   'dateTime',
   'decimal',
   'id',
   'instant',
   'integer',
   'markdown',
   'oid',
   'positiveInt',
   'string',
   'time',
   'unsignedInt',
   'uri',
   'url',
   'uuid',
   'Address',
   'Age',
   'Annotation',
   'Attachment',
   'CodeableConcept',
   'Coding',
   'ContactPoint',
   'Count',
   'Distance',
   'Duration